<h1><center>Laboratorio 9: Optimización de modelos 💯</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2024</strong></center>

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebastián Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicolás Ojeda, Melanie Peña, Valentina Rojas

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Renato Pino
- Nombre de alumno 2: Valentina Abello


### **Link de repositorio de GitHub:** [Repositorio](https://github.com/Renato-98/MDS7202-1)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.

### Reglas:

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.
- Código que no se pueda ejecutar, no será revisado.

### Objetivos principales del laboratorio

- Optimizar modelos usando `optuna`
- Recurrir a técnicas de *prunning*
- Forzar el aprendizaje de relaciones entre variables mediante *constraints*
- Fijar un pipeline con un modelo base que luego se irá optimizando.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [ ]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [ ]:
# Si usted está utilizando Colabolatory le puede ser útil este código para cargar los archivos.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    path = '/content/drive/MyDrive/Lab_progra_DS/Lab_9/' # Ajustar al path propio
except:
    print('Ignorando conexión drive-colab')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv(path + 'sales.csv')
df.sample(10)

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
375,383,31/05/12,Patra,38.24444,21.73444,164250,shop_6,adult-cola,glass,500ml,1.33,28775
3488,3540,30/04/15,Patra,38.24444,21.73444,167001,shop_6,gazoza,glass,500ml,0.55,91516
3011,3056,31/10/14,Athens,37.96245,23.68708,668203,shop_3,lemon-boost,glass,500ml,1.18,9846
5498,5588,31/03/17,Athens,37.97945,23.71622,665871,shop_1,gazoza,glass,500ml,0.78,24310
2047,2078,31/12/13,Athens,37.97945,23.71622,671022,shop_1,gazoza,can,330ml,0.30,43629
4557,4624,30/04/16,Patra,38.24444,21.73444,168254,shop_6,lemon-boost,plastic,1.5lt,2.03,28491
5756,5847,31/05/17,Irakleion,35.32787,25.14341,138200,shop_2,gazoza,glass,500ml,0.85,22186
3541,3594,30/04/15,Patra,38.24444,21.73444,167001,shop_6,orange-power,can,330ml,0.56,49715
1773,1798,31/08/13,Patra,38.24444,21.73444,166301,shop_6,orange-power,plastic,1.5lt,2.17,25801
4430,4497,29/02/16,Patra,38.24444,21.73444,168254,shop_6,orange-power,plastic,1.5lt,1.97,21867


**Nota:** Las fechas estan en formato dia-mes-año

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7456 entries, 0 to 7455
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         7456 non-null   int64  
 1   date       7456 non-null   object 
 2   city       7456 non-null   object 
 3   lat        7456 non-null   float64
 4   long       7456 non-null   float64
 5   pop        7456 non-null   int64  
 6   shop       7456 non-null   object 
 7   brand      7456 non-null   object 
 8   container  7456 non-null   object 
 9   capacity   7456 non-null   object 
 10  price      7456 non-null   float64
 11  quantity   7456 non-null   int64  
dtypes: float64(3), int64(3), object(6)
memory usage: 699.1+ KB


In [ ]:
df.describe()

,id,lat,long,pop,price,quantity
count,7456.000000,7456.000000,7456.000000,7456.000000,7456.000000,7456.000000
mean,3784.926770,38.300616,23.270170,355042.733637,1.197193,29408.428380
std,2185.822361,1.650030,1.086592,232336.703020,0.818175,17652.985675
min,0.000000,35.327870,21.734440,134219.000000,0.110000,2953.000000
25%,1889.750000,37.962450,22.417610,141732.000000,0.620000,16572.750000
50%,3783.500000,38.244440,22.930860,257501.500000,0.930000,25294.500000
75%,5682.250000,39.636890,23.716220,665102.000000,1.510000,37699.000000
max,7559.000000,40.643610,25.143410,672130.000000,4.790000,145287.000000


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [ ]:
from sklearn import set_config
set_config(transform_output="pandas")

#1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad.
from sklearn.model_selection import train_test_split

# Fijamos una semilla para la reproducibilidad
seed = 42

# Separacion inicial: 70% train, 30% temp
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=seed)

# Separación de temp en validation (20%) y test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.33, random_state=seed)  # 33% de 30% es ~10%

# Verificación de tamaños
print(f"Tamaño de train: {len(train_df)}")
print(f"Tamaño de validation: {len(val_df)}")
print(f"Tamaño de test: {len(test_df)}")


Tamaño de train: 5219
Tamaño de validation: 1498
Tamaño de test: 739


In [ ]:
# Separar las variables predictoras (X) y la variable objetivo (y)
#Train
X_train = train_df.drop(columns=['quantity'])
y_train = train_df['quantity']
#Validacion
X_val = val_df.drop(columns=['quantity'])
y_val = val_df['quantity']
#Test
X_test = test_df.drop(columns=['quantity'])
y_test = test_df['quantity']

In [ ]:
#2. Implemente un FunctionTransformer para extraer el día, mes y año de la variable date.
# Guarde estas variables en el formato categorical de pandas.

from sklearn.preprocessing import FunctionTransformer

# Función que extrae día, mes y año especificando el formato de la fecha
def extract_date_components(df):
    # Especificamos el formato correcto de la columna 'date'
    date_format = '%d/%m/%y'  # El formato en el que parecen estar las fechas
    df['date'] = pd.to_datetime(df['date'], format=date_format, errors='coerce')  # 'errors="coerce"' para manejar fechas invalidas
    df['year'] = df['date'].dt.year.astype('category')
    df['month'] = df['date'].dt.month.astype('category')
    df['day'] = df['date'].dt.day.astype('category')
    return df.drop(columns=['date'])

# Creacion del FunctionTransformer
date_transformer = FunctionTransformer(extract_date_components)

# Aplicar el FunctionTransformer para extraer año, mes y día a X_train y X_val
X_train_transformed = date_transformer.transform(X_train.copy())
X_val_transformed = date_transformer.transform(X_val.copy())

# Verificamos las nuevas columnas
X_train_transformed.head()

,id,city,lat,long,pop,shop,brand,container,capacity,price,year,month,day
292,300,Patra,38.24444,21.73444,164250,shop_6,adult-cola,plastic,1.5lt,2.54,2012,4,30
3366,3416,Athens,37.97945,23.71622,667237,shop_1,gazoza,plastic,1.5lt,0.71,2015,2,28
3685,3741,Athens,37.96245,23.68708,667237,shop_3,adult-cola,can,330ml,0.66,2015,6,30
2404,2441,Athens,37.97945,23.71622,668203,shop_1,gazoza,can,330ml,0.30,2014,4,30
2855,2898,Irakleion,35.32787,25.14341,136202,shop_2,orange-power,can,330ml,0.56,2014,9,30


In [ ]:
#3. Implemente un ColumnTransformer para procesar de manera adecuada los datos numéricos y categóricos.
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Definir las columnas categóricas y numéricas
categorical_columns = ['year', 'month', 'day', 'city', 'shop', 'brand', 'container', 'capacity']
numerical_columns = ['lat', 'long', 'pop', 'price', 'quantity']

# Crear un Cclumntransformer para aplicar el escalado y onehotencoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),  # Escalar las variables numericas
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)  # onehotencoder para las categiricas
    ],
    remainder='passthrough'  # Mantener cualquier otra columna sin transformar
)

# Configuramos la salida en formato DataFrame de pandas
from sklearn import set_config
set_config(transform_output="pandas")


In [ ]:
#4. Guarde los pasos anteriores en un Pipeline, dejando como último paso el regresor DummyRegressor para generar predicciones en base a promedios.
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# Definir las columnas categóricas y numéricas
categorical_columns = ['year', 'month', 'day', 'city', 'shop', 'brand', 'container', 'capacity']
numerical_columns = ['lat', 'long', 'pop', 'price']

# Crear un ColumnTransformer para aplicar el escalado y OneHotEncoder sin salida esparsa
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),  # Escalar las variables numéricas
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns)  # OneHotEncoder sin salida esparsa
    ],
    remainder='passthrough'  # Mantener cualquier otra columna sin transformar
)

# Crear el pipeline con el preprocesador y el DummyRegressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

#5.
# Entrenamos el pipeline con los datos ya transformados
pipeline.fit(X_train_transformed, y_train)

# Realizamos predicciones en el conjunto de validación
val_predictions = pipeline.predict(X_val_transformed)

# Calcular el MAE (Mean Absolute Error)
mae = mean_absolute_error(y_val, val_predictions)
print(f'MAE del DummyRegressor: {mae}')


MAE del DummyRegressor: 13308.134750658153


In [ ]:
#6.
from xgboost import XGBRegressor

# Pipeline con el preprocesador y el XGBRegressor
pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state= seed))
])

# Entrenar el pipeline
pipeline_xgb.fit(X_train_transformed, y_train)

# Realizar predicciones
val_predictions_xgb = pipeline_xgb.predict(X_val_transformed)

# Calcular el MAE (Mean Absolute Error)
mae_xgb = mean_absolute_error(y_val, val_predictions_xgb)
print(f'MAE del XGBRegressor: {mae_xgb}')


MAE del XGBRegressor: 2402.1883264600197


In [ ]:
13308.134750658153/2402.1883264600197

5.540004754860191

El valor del MAE par Xgboost es alrededor de 5.5 veces menor que el DummyRegressor, por lo que claramente es un modelo muy superior.

In [ ]:
#7. Guarde ambos modelos en un archivo .pkl (uno cada uno)
import joblib

# Guardar el modelo DummyRegressor
joblib.dump(pipeline, 'dummy_regressor.pkl')

# Guardar el modelo XGBRegressor
joblib.dump(pipeline_xgb, 'xgb_regressor.pkl')

print("Modelos guardados correctamente.")

Modelos guardados correctamente.


## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [ ]:
#1.
# Ajustamos el OneHotEncoder para mantener pandas y no usar matrices sparse com nos sugiere el hint 1
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns)
    ],
    remainder='passthrough'  # Mantener el formato pandas en las demas columnas
)

# Aplicar el FunctionTransformer para extraer año, mes y día
X_train_transformed = date_transformer.transform(X_train.copy())
Y_val_transformed = date_transformer.transform(X_val.copy())

# Ajustar el preprocesador para entrenar y obtener los nombres de las columnas
preprocessor.fit(X_train_transformed)
feature_names = preprocessor.get_feature_names_out()

# Verificar como se llama la columna 'price' despues del preprocesamiento
price_feature_name = [name for name in feature_names if 'price' in name][0]
print(f'Nombre de la columna "price" después del preprocesamiento: {price_feature_name}')

# Crear el pipeline con la restricción monótona aplicada a la columna procesada
pipeline_xgb_monotonic = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=seed, monotone_constraints={price_feature_name: -1}))
])

# Entrenar el pipeline con los datos transformados en formato pandas
pipeline_xgb_monotonic.fit(X_train_transformed, y_train)

#2.
# Realizar predicciones en el conjunto de validación
val_predictions_xgb_monotonic = pipeline_xgb_monotonic.predict(X_val_transformed)

# Calcular el MAE (Mean Absolute Error)
mae_xgb_monotonic = mean_absolute_error(y_val, val_predictions_xgb_monotonic)
print(f'MAE del XGBRegressor con restricciones monótonas (nombre ajustado): {mae_xgb_monotonic}')


Nombre de la columna "price" después del preprocesamiento: num__price
MAE del XGBRegressor con restricciones monótonas (nombre ajustado): 2493.6477391404687


3. El MAE reportado aumento ligeramente de 2402.18 a 2493.65, lo que sugiere que la restricción monotona negativa no mejora el modelo en este caso.

  Por lo tanto, la relacion monotona negativa no parece beneficiar al modelo. Es posible que la relacion entre precio y cantidad sea mas compleja de lo que el colega habia anticipado, y limitarla a ser monotona negativa podria estar impidiendo que el modelo capture correctamente todos los patrones.

In [ ]:
#4.
# Guardar el modelo XGBRegressor
joblib.dump(pipeline_xgb_monotonic, 'xgb_monotonic.pkl')

print("Modelos guardados correctamente.")

Modelos guardados correctamente.


## 3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
#!pip install optuna

In [ ]:
import optuna
from optuna.samplers import TPESampler
# Inserte su código acá
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

#1.
def objective(trial):
    # Hiperparametros de XGBRegressor
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1)
    n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    max_leaves = trial.suggest_int('max_leaves', 0, 100)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 5)
    reg_alpha = trial.suggest_float('reg_alpha', 0.0, 1.0)
    reg_lambda = trial.suggest_float('reg_lambda', 0.0, 1.0)

    # Hiperparametro del OneHotEncoder
    min_frequency = trial.suggest_float('min_frequency', 0.0, 1.0)

    # Ajustamos el OneHotEncoder con el min_frequency sugerido
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_columns),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=min_frequency), categorical_columns)
        ],
        remainder='passthrough'
    )

    # Crear el pipeline
    pipeline_xgb_optuna = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor(
            random_state=seed,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_leaves=max_leaves,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda
        ))
    ])

    # Entrenar el pipeline con los datos transformados
    pipeline_xgb_optuna.fit(X_train_transformed, y_train)

    # Realizar predicciones en el conjunto de validación
    val_predictions_optuna = pipeline_xgb_optuna.predict(X_val_transformed)

    # Calcular el MAE
    mae = mean_absolute_error(y_val, val_predictions_optuna)

    # Almacenar el mejor pipeline en el trial utilizando set_user_attr
    trial.set_user_attr("best_pipeline", pipeline_xgb_optuna)

    return mae

In [ ]:
#2.
# Configurar el sampler de Optuna
sampler = TPESampler(seed=seed)

# Crear el estudio de Optuna
study = optuna.create_study(direction='minimize', sampler=sampler)

# Ejecutar la optimizacion durante 5 minutos (300 segundos)
study.optimize(objective, timeout=300)

# Imprimir los mejores hiperparametros encontrados
print(f'Mejor MAE: {study.best_value}')
print(f'Mejores hiperparámetros: {study.best_params}')

[I 2024-10-24 23:40:26,974] A new study created in memory with name: no-name-38f3d127-de9d-4713-80d3-ce09f69bb0c2
[I 2024-10-24 23:40:28,554] Trial 0 finished with value: 6988.087557824495 and parameters: {'learning_rate': 0.03807947176588889, 'n_estimators': 954, 'max_depth': 8, 'max_leaves': 60, 'min_child_weight': 1, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946, 'min_frequency': 0.8661761457749352}. Best is trial 0 with value: 6988.087557824495.
[I 2024-10-24 23:40:29,169] Trial 1 finished with value: 3492.0901505555585 and parameters: {'learning_rate': 0.06051038616257767, 'n_estimators': 723, 'max_depth': 3, 'max_leaves': 97, 'min_child_weight': 5, 'reg_alpha': 0.21233911067827616, 'reg_lambda': 0.18182496720710062, 'min_frequency': 0.18340450985343382}. Best is trial 1 with value: 3492.0901505555585.
[I 2024-10-24 23:40:29,831] Trial 2 finished with value: 6636.115521134298 and parameters: {'learning_rate': 0.03111998205299424, 'n_estimators': 549, 'max_dep

Mejor MAE: 2021.2459897702145
Mejores hiperparámetros: {'learning_rate': 0.05258525231231221, 'n_estimators': 678, 'max_depth': 10, 'max_leaves': 79, 'min_child_weight': 4, 'reg_alpha': 0.5332902675779876, 'reg_lambda': 0.3305773756436482, 'min_frequency': 0.029251484155416785}


In [ ]:
#3.
# Reportar los resultados de la optimización
print(f'Número de trials: {len(study.trials)}')
print(f'Mejor MAE obtenido: {study.best_value}')
print(f'Mejores hiperparámetros: {study.best_params}')

# Acceder al mejor pipeline
best_trial = study.best_trial
best_pipeline = best_trial.user_attrs["best_pipeline"]

Número de trials: 133
Mejor MAE obtenido: 2021.2459897702145
Mejores hiperparámetros: {'learning_rate': 0.05258525231231221, 'n_estimators': 678, 'max_depth': 10, 'max_leaves': 79, 'min_child_weight': 4, 'reg_alpha': 0.5332902675779876, 'reg_lambda': 0.3305773756436482, 'min_frequency': 0.029251484155416785}


3. Vemos que el MAE ha disminuido considerablemente desde los experimentos anteriores:

* Sin restricciones: 2402.18
* Con restricciones monótonas: 2493.64
* Con Optuna: 2039.46

El nuevo MAE es 362.72 puntos mas bajo que el mejor modelo anterior, lo que indica una mejora significativa en la capacidad predictiva del modelo.

El cambio se debe a la optimización de hiperparametros con Optuna, que ha permitido encontrar una configuración mas adecuada de hiperparametros para el modelo XGBRegressor y el preprocesador OHE, esto lo hizo explorado un amplio espacio de hiperparametros, encontrando la mejor combinacion posible para este conjunto de datos.

Ademas Optuna usa un enfoque bayesiano para optimizar, lo que significa que cada prueba aprovecha la información de los intentos anteriores. Esto lo hace más eficiente que una simple búsqueda aleatoria o en cuadricula como lo es GridSearchCV de scikit-learn.

4.  Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados?

* Leaning rate:

El learning rate controla la tasa de aprendizaje del modelo, es decir, que tanto se ajustan los pesos de los arboles en cada iteracion. Valores más bajos permiten que el modelo aprenda mas lentamente pero con mayor precision. Un valor bajo de learning_rate hace que el modelo sea más robusto pero mas lento para entrenar, mientras que un valor alto podria hacer que el modelo aprenda patrones demasiado raido, lo que puede llevar a sobreajuste.

El rango 0.001 a 0.1 es comunmente utilizado para optimizar la tasa de aprendizaje en XGBoost. El valor default de este es de 0.36.

---

* n_estimators:

Indica el numero de arboles a construir en el ensamble de XGBoost. Mas arboles generalmente ayudan al modelo a aprender mejor, pero tambien aumentan el tiempo de entrenamiento y el riesgo de sobreajuste si el numero es muy alto. Por tanto, un mayor numero de estimadores puede mejorar el rendimiento, pero hay un punto en el que añadir mas arboles ya no aporta mejora significativa.

El rango 50 - 1000 es razonable. Menos de 50 qrboles puede ser insuficiente para capturar patrones complejos, mientras que mas de 1000 puede aumentar el riesgo de sobreajuste y el costo computacional sin mejorar mucho el rendimiento.

---
* max_depth:

Define la profundidad maxima que puede tener cada arbol. Los arboles mass profundos son mas capaces de capturar relaciones complejas en los datos, pero tambien son mas propensos a sobreajustar. Por lo tanto, este parametro controla la complejidad del modelo.

El rango de 3 a 10 es adecuado cosidrando que el default es de 6. Es comun comenzar con un valor pequeño, como max_depth = 3 y aumentarlo hasta que el rendimiento en el conjunto de validación deje de mejorar.

---

* max_leaves:

Este parametro limita el numero máximo de hojas en cada arbol. Recordemos que las hojas representan nodos terminales que definen las reglas de predicción del arbol. Asi, este hiperparametro actua como una forma de regular la complejidad de los arboles. Un numero menor de hojas puede simplificar los arboles, mientras que un numero mayor permite arboles mas complejos y detallados.

El rango de 0 a 100 es razonable. Un valor de 0 indica que no se impone ninguna restricción, lo que puede resultar en un consumo elevado de memoria durante el entrenamiento.

---

* min_child_weight:

Se encarga de especificar el peso mínimo de las instancias que deben estar presentes en un nodo hijo para que se permita la división. Este valor actúa como un regularizador ya que evita que el modelo divida un nodo si no hay suficientes datos en una rama, lo que ayuda a reducir el sobreajuste.

El rango de 1 a 5 es parece ser comun. Los valores mas bajos permiten mas divisiones, lo que puede ser util para modelos mas complejos, mientras que valores mas altos evitan divisiones excesivas.

---

* reg_alpha:

Controla la regularización L1 (Lasso) que penaliza los pesos del modelo. Ayuda a reducir el sobreajuste forzando a que algunos pesos sean 0.

El rango de 0 a 1 es comun. Un valor de 0 significa que no hay regularización, mientras que valores más altos aplican mayor penalización.

---

* reg_lambda:

Controla la regularización L2 (Ridge), que penaliza los pesos grandes del modelo. Tambien reduce el sobreajuste, pero lo hace suavizando los pesos en lugar de eliminarlos.

Igaulemnte, el el rango de 0 a 1 es adecuado. Un valor de 0 significa que no hay regularizacion, mientras que un valor cercano a 1 aplica una regularizacion mas fuerte.

---

* min_frequency:

Este hiperparametro controla el umbral minimo de frecuencia para incluir categorias en la codificación One-Hot. Si una categoria aparece con una frecuencia menor al valor indicado, se agrupa en una categoría "otros". Por tanto, este hiperparametro reduce la dimensionalidad del modelo, agrupando categorías raras en una unica categoría, lo que puede mejorar la capacidad de generalización del modelo.

Un rango de 0 a 1 es logico. Un valor cercano a 0  incluye más categorías (todas para exactamente 0), mientras que un valor cercano a 1 agrupa más categorías raras.

---

Referencias:

* https://xgboost.readthedocs.io/en/stable/parameter.html
* https://medium.com/@rithpansanga/the-main-parameters-in-xgboost-and-their-effects-on-model-performance-4f9833cac7c

In [ ]:
#5.
import joblib

# Guardar el mejor pipeline
joblib.dump(best_pipeline, 'xgb_regressor_optuna_best_pipeline.pkl')

print("Mejor pipeline guardado correctamente.")

Mejor pipeline guardado correctamente.


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

1. Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
2. Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
3. Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
4. Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [ ]:
!pip install optuna-integration[xgboost]

**¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento?**
Prunning es una técnica que se utiliza en el proceso de optimización para detener tempranamente las pruebas de los modelos cuando no se están obteniendo buenos resultados, basándose en su desempeño hasta ese momento. Es útil para cuando se están comparando muchos modelos y se busca encontrar los mejores hiperparámetros.

Impacta en el entramiento porque reduce su tiempo de entranamiento, evita el sobreajuste al no realizar iteracciones innecesarias y de esta forma se optimizan los recursos.

In [ ]:
import optuna
from optuna.integration import XGBoostPruningCallback

def objective(trial):
    # Sugerencias de hiperparámetros para XGBRegressor
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1)
    n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    max_leaves = trial.suggest_int('max_leaves', 0, 100)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 5)
    reg_alpha = trial.suggest_float('reg_alpha', 0.0, 1.0)
    reg_lambda = trial.suggest_float('reg_lambda', 0.0, 1.0)


    # Hiperparámetro del OneHotEncoder
    min_frequency = trial.suggest_float('min_frequency', 0.0, 1.0)

    # Ajuste del preprocesador (OneHotEncoder con min_frequency sugerido)
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_columns),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=min_frequency), categorical_columns)
        ],
        remainder='passthrough'
    )

    # Crear el pipeline
    pipeline_xgb_optuna_prunner = Pipeline(steps=[
        ('preprocessor', preprocessor),
         ('regressor', XGBRegressor(
             random_state=seed,
             learning_rate=learning_rate,
             n_estimators=n_estimators,
             max_depth=max_depth,
            max_leaves=max_leaves,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda
        ))
    ])

    # Entrenar el pipeline con los datos transformados y agregar el pruning callback
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-mae")
    pipeline_xgb_optuna_prunner.fit(X_train_transformed, y_train)

    # Realizar predicciones en el conjunto de validación
    val_predictions_optuna_prunner = pipeline_xgb_optuna_prunner.predict(X_val_transformed)

    # Calcular el MAE
    mae = mean_absolute_error(y_val, val_predictions_optuna_prunner)

    # Almacenar el mejor pipeline en el trial utilizando set_user_attr
    trial.set_user_attr("best_pipeline", pipeline_xgb_optuna_prunner)

    return mae

In [ ]:
# 3. Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
study.optimize(objective, timeout=300)

[I 2024-10-24 23:50:57,673] Trial 139 finished with value: 2077.7754905303427 and parameters: {'learning_rate': 0.04488050572581134, 'n_estimators': 642, 'max_depth': 10, 'max_leaves': 86, 'min_child_weight': 5, 'reg_alpha': 0.42267576796691697, 'reg_lambda': 0.3924223168127448, 'min_frequency': 0.02038057914905607}. Best is trial 131 with value: 2021.2459897702145.
[I 2024-10-24 23:50:59,689] Trial 140 finished with value: 2068.585181772311 and parameters: {'learning_rate': 0.05869812772838082, 'n_estimators': 690, 'max_depth': 10, 'max_leaves': 71, 'min_child_weight': 4, 'reg_alpha': 0.4732212508384608, 'reg_lambda': 0.4321219320669366, 'min_frequency': 0.054340557401213616}. Best is trial 131 with value: 2021.2459897702145.
[I 2024-10-24 23:51:02,221] Trial 141 finished with value: 2083.2055636356286 and parameters: {'learning_rate': 0.0416082933872461, 'n_estimators': 807, 'max_depth': 10, 'max_leaves': 82, 'min_child_weight': 4, 'reg_alpha': 0.5863043952355497, 'reg_lambda': 0.264

In [ ]:
# 4. Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
print(f'Número de trials: {len(study.trials)}')
print(f'Mejor MAE obtenido: {study.best_value}')
print(f'Mejores hiperparámetros: {study.best_params}')

Número de trials: 229
Mejor MAE obtenido: 2000.9103635576603
Mejores hiperparámetros: {'learning_rate': 0.06735520736182712, 'n_estimators': 889, 'max_depth': 10, 'max_leaves': 84, 'min_child_weight': 4, 'reg_alpha': 0.8198420225145645, 'reg_lambda': 0.05701224506872668, 'min_frequency': 0.020194403319583366}


*   **Optuna:**
```
Número de trials: 124
Mejor MAE obtenido: 2039.4655323308682
Mejores hiperparámetros: {'learning_rate': 0.06258057837647998, 'n_estimators': 800,
'max_depth': 10, 'max_leaves': 78, 'min_child_weight': 4, 'reg_alpha': 0.7627014672965421,
'reg_lambda': 0.2529218490083219, 'min_frequency': 0.011606578881046742}
```
*   **Optuna y Prunning:**
```
Número de trials: 202
Mejor MAE obtenido: 2008.5927721336782
Mejores hiperparámetros: {'learning_rate': 0.06737610202159762, 'n_estimators': 1000,
'max_depth': 10, 'max_leaves': 84, 'min_child_weight': 4, 'reg_alpha': 0.1968005888173449,
'reg_lambda': 0.21615935883382276, 'min_frequency': 0.0204794515229899}
```

El nuevo **MAE** con Optuna y Prunning es de **2008.59**, siendo levemente menor que en la sección anterior, esta mejora muestra un ajuste más preciso de los hiperparámetros al incluir la **Prunning**, que permite detener los trials que no presentan una mejora clara, optimizando así su tiempo de entrenamiento.

El número **trials** aumentó a **202**, este aumento se debe a que, al eliminar combinaciones menos efectivas durante el entrenamiento del modelo, Optuna logra explorar más configuraciones en un tiempo similar que a la sección anterior, teniendo más oportunidades de encontrar una combinación de hiperparámetros mejor.

Los **hiperparámetros** son similares en los dos modelos. Se ve un cambio significativo en `reg_alpha`, que disminuyó de **0.7627** a **0.1968**, que indica menos penalización en los coeficientes, permitiendo que el modelo se ajuste mejor a nuestros datos.

In [ ]:
# 5. Guardar su modelo en un archivo .pkl [1 punto]
best_pipeline_prunning = study.best_trial.user_attrs["best_pipeline"]
joblib.dump(best_pipeline_prunning, 'xgb_regressor_optuna_prunning_best_pipeline.pkl')

print("Mejor pipeline guardado correctamente.")

Mejor pipeline guardado correctamente.


## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
!pip install plotly

In [ ]:
#1. Gráfico de historial de optimización [1 punto]
import optuna.visualization as vis

vis.plot_optimization_history(study)

In [ ]:
#2. Gráfico de coordenadas paralelas [1 punto]
vis.plot_parallel_coordinate(study)

In [ ]:
#3. Gráfico de importancia de hiperparámetros [1 punto]
vis.plot_param_importances(study)

**¿Desde qué trial se empiezan a observar mejoras notables en sus resultados?**

Del **Gráfico de historial de optimización**, se observa que las mejoras notables se empiezan a observar desde los primers trials, se podría decir que desde el 13 aproximadamente, donde el valor objetivo (`mae`) empieza a caer notablemente para luego mantenerse cerca del 2.000.

**¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas?**

Del **Gráfico de coordenadas paralelas** se puede observar que el hiperparámetro `min_frequency` tiene una fuerte correlación con los resultados, de manera similar ocurre con `reg_lambda` y `n_estimators` pero no tan fuerte como con el anterior.

**¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo?**

Del **Gráfico de importancia de hiperparámetros** se observa que el hiperparámetro `min_frequency` es el más importante para la optimización del modelo, con una importancia del 85%. Luego, siguen ``reg_lambda` y `n_estimators` con 7% y 4%, respectivamente, similiar a la conslusión anterior.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
mae_xgb_optuna = mean_absolute_error(y_val, best_pipeline.predict(X_val_transformed))
mae_xgb_pruning = mean_absolute_error(y_val, best_pipeline_prunning.predict(X_val_transformed))


MAE XGBoost con Optuna: 2021.2459897702145
MAE XGBoost con Prunning: 2000.9103635576603


In [ ]:
mae_xgb_optuna = mean_absolute_error(y_val, best_pipeline.predict(X_val_transformed))
mae_xgb_pruning = mean_absolute_error(y_val, best_pipeline_prunning.predict(X_val_transformed))

mae_tabla = {
    'Modelo': ['Baseline', 'XGBoost', 'XGBoost con Restricciones Monótonas', 'XGBoost con Optuna', 'XGBoost con Optuna y Prunning'],
    'MAE': [mae, mae_xgb, mae_xgb_monotonic, mae_xgb_optuna, mae_xgb_pruning]
}
mae_tabla = pd.DataFrame(mae_tabla)
mae_tabla

,Modelo,MAE
0,Baseline,13308.134751
1,XGBoost,2402.188326
2,XGBoost con Restricciones Monótonas,2493.647739
3,XGBoost con Optuna,2021.245990
4,XGBoost con Optuna y Prunning,2000.910364


De la tabla, se puede observar que el modelo que tiene mejor rendimiento es el **XGBoost con Optuna y Prunning** con un **MAE** de **2000.91**

In [ ]:
# 2. Predecir sobre el conjunto de test
X_test_transformed = date_transformer.transform(X_test.copy())

# Predecir sobre el conjunto de test
test_predictions = best_pipeline_prunning.predict(X_test_transformed)

# Calcular el MAE en el conjunto de test
mae_test = mean_absolute_error(y_test, test_predictions)

print(f'MAE en el conjunto de test: {mae_test}')

MAE en el conjunto de test: 2123.9053279479235


**¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto?**

El MAE en el **conjunto de test** fue de **2123.90**, mientras que en el **conjunto de validación** fue de **2000.91**, esto se puede deber a varias razones, como la distribución de los datos, el tamaño que se utilizó para definir los conjuntos de entrenamiento, test y validación. Esta diferencia es "normal" en estas situaciones, por último, la diferencia no es relativamente grande y podría estar bien de acuerdo a las razones dadas anteriormente.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media.tenor.com/8CT1AXElF_cAAAAC/gojo-satoru.gif">
</p>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=87110296-876e-426f-b91d-aaf681223468' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>